# Notebook 05 — Computational Analysis
## Monte Carlo Simulation · Bloom Filter · MCMC Knapsack

---

| Field | Detail |
|---|---|
| **Member** | Muhammad Fadhil |
| **Role** | Member E — Computation Analyst |
| **Repository** | `microsoft/vscode` |

---

## Research Question (Layer 3 — Simulation)

> **RQ3:** Berapakah peluang sebuah issue acak di repositori `microsoft/vscode`
> akan tetap terbuka lebih lama dari **12 hari**, jika dihitung menggunakan
> simulasi Monte Carlo tanpa asumsi distribusi?

---

### Keterkaitan dengan Layer Sebelumnya

| Layer | Anggota | Temuan Kunci yang Digunakan di Notebook Ini |
|---|---|---|
| EDA | Member A | `days_open` max=12, skewness=**1.6913** → right-skewed → justifikasi MC. Baseline empiris P(>12 hr)=**0.0000** |
| Estimation | Member B | MLE Bernoulli θ̂=**0.8723** (k=1134, n=1300) |
| Confidence Interval | Member C | CI 95% merge rate sempit → estimasi stabil dan representatif |
| Hypothesis Testing | Member D | Z=**5.4433**, p≈**0.0000** → **Tolak H₀**. Merge rate User (0.8986) ≠ Bot (0.7045) |

## AI Usage Disclosure

**Member:** Muhammad Fadhil — Computation Analyst &nbsp;|&nbsp; **Tools used:** Claude

| # | Task | Tool | Prompt summary | Output modified? |
|---|---|---|---|---|
| 1 | Scaffold notebook sesuai data aktual kelompok | Claude | "buatkan 05_simulation siap jalan di VS Code" | Ya — interpretasi & kesimpulan ditulis sendiri |

**Ditulis sepenuhnya tanpa AI:**
- Semua cell `### Interpretasi`
- Research question di header
- Kesimpulan akhir (Summary cell)

---
## 0. Setup & Imports

In [1]:
import sys
import os
import random
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

# ── Tambahkan root project ke sys.path ────────────────────────────────────────
# Struktur folder yang dibutuhkan:
#
#   project_root/
#   ├── src/
#   │   ├── __init__.py        ← buat file kosong ini!
#   │   └── simulation.py
#   ├── data/
#   │   └── clean/
#   │       ├── issues_clean.csv
#   │       └── pull_requests_clean.csv
#   ├── notebooks/
#   │   └── 05_simulation.ipynb  ← file ini
#   └── report/                  ← dibuat otomatis

ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

from src.simulation import estimate_probability, BloomFilter, mcmc_knapsack

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Import berhasil.')
print(f'ROOT : {ROOT}')

ModuleNotFoundError: No module named 'src'

---
## 1. Load Data

In [ ]:
DATA_DIR = os.path.join(ROOT, 'data', 'clean')

df_issues = pd.read_csv(
    os.path.join(DATA_DIR, 'issues_clean.csv'),
    parse_dates=['created_at', 'updated_at', 'closed_at']
)
df_pulls = pd.read_csv(
    os.path.join(DATA_DIR, 'pull_requests_clean.csv'),
    parse_dates=['created_at', 'updated_at', 'closed_at', 'merged_at']
)

print(f'Issues       : {len(df_issues):,} baris, {len(df_issues.columns)} kolom')
print(f'Pull Requests: {len(df_pulls):,} baris, {len(df_pulls.columns)} kolom')

# ── Angka kunci dari layer sebelumnya ─────────────────────────────────────────
THETA_HAT  = 0.8723   # MLE Bernoulli keseluruhan  — Member B
THETA_USER = 0.8986   # merge rate User             — Member D
THETA_BOT  = 0.7045   # merge rate Bot              — Member D
Z_STAT     = 5.4433   # Z-statistic                 — Member D
SKEWNESS   = 1.6913   # skewness days_open          — Member A
P_EMPIRIS  = 0.0000   # P(days_open > 12) empiris  — Member A

print()
print(f'{"θ̂ keseluruhan  (Member B)":<30}: {THETA_HAT}')
print(f'{"θ̂ User         (Member D)":<30}: {THETA_USER}')
print(f'{"θ̂ Bot          (Member D)":<30}: {THETA_BOT}')
print(f'{"Z-statistic    (Member D)":<30}: {Z_STAT}  → Tolak H₀')
print(f'{"Skewness days_open (A)":<30}: {SKEWNESS}')
print(f'{"P(days_open > 12)  (A)":<30}: {P_EMPIRIS}')

df_issues[['days_open','comments_count','label_count']].describe().round(2)

---
## 2. Monte Carlo Simulation — RQ3

### Latar Belakang & Justifikasi Metode

**Mengapa Monte Carlo?**
Member A (EDA) menemukan bahwa distribusi `days_open` sangat **right-skewed**
(skewness = 1.6913, mean = 1.68, max = 12 hari).
Karena distribusi tidak normal, formula analitik berbasis asumsi normalitas
akan menghasilkan estimasi yang **bias**.
Monte Carlo tidak memerlukan asumsi distribusi — ia langsung
menggunakan data empiris sebagai dasar sampling.

**Formula (Tsun, 2020, p. 314–315):**

$$\hat{P}(\text{event}) = \frac{\text{jumlah\_sukses}}{N_{\text{trials}}}$$

$$SE = \sqrt{\frac{\hat{p}\,(1 - \hat{p})}{N_{\text{trials}}}}$$

**Threshold:** 12 hari (sesuai RQ3)

**Baseline empiris Member A:** P(days\_open > 12) = **0.0000**
(seluruh 590 issue diselesaikan dalam ≤ 12 hari)

In [ ]:
# ── Persiapan data ─────────────────────────────────────────────────────────
valid_days = df_issues['days_open'].dropna()
valid_days = valid_days[valid_days >= 0].values

THRESHOLD = 12   # sesuai RQ3

# Statistik deskriptif days_open
print('=== Statistik days_open (590 issues) ===')
print(f'  Min      : {valid_days.min():.0f} hari')
print(f'  Max      : {valid_days.max():.0f} hari')
print(f'  Mean     : {valid_days.mean():.4f} hari')
print(f'  Std      : {valid_days.std():.4f}')
print(f'  Skewness : {pd.Series(valid_days).skew():.4f}')
print(f'  P(> {THRESHOLD} hari) empiris: {(valid_days > THRESHOLD).mean():.4f}')

# ── Event function ────────────────────────────────────────────────────────
def event_issue_slow():
    """Satu trial Monte Carlo: ambil 1 issue acak, cek apakah > THRESHOLD hari."""
    return np.random.choice(valid_days) > THRESHOLD

# ── Jalankan Monte Carlo (src/simulation.py) ─────────────────────────────
mc_result = estimate_probability(event_fn=event_issue_slow, n_trials=50_000)

print()
print('=== Hasil Monte Carlo (n = 50.000 trials) ===')
print(f"  P̂(issue > {THRESHOLD} hari) : {mc_result['p_hat']:.4f}")
print(f"  Standard Error           : {mc_result['std_error']:.6f}")
print(f"  Jumlah Sukses            : {mc_result['successes']:,} / {mc_result['n_trials']:,}")
print(f"  Konsisten dgn empiris    : {'YA ✓' if abs(mc_result['p_hat'] - P_EMPIRIS) < 0.005 else 'PERIKSA'}")

In [ ]:
# ── Konvergensi Monte Carlo ────────────────────────────────────────────────
checkpoints  = [100, 500, 1_000, 5_000, 10_000, 25_000, 50_000]
p_hats_conv  = [
    estimate_probability(event_fn=event_issue_slow, n_trials=n)['p_hat']
    for n in checkpoints
]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Kiri — kurva konvergensi
axes[0].plot(checkpoints, p_hats_conv, marker='o', linewidth=2,
             color='steelblue', label='Estimasi Monte Carlo')
axes[0].axhline(mc_result['p_hat'], color='tomato', linestyle='--',
                label=f"Final (n=50k): {mc_result['p_hat']:.4f}")
axes[0].axhline(P_EMPIRIS, color='seagreen', linestyle=':',
                label=f'Baseline empiris: {P_EMPIRIS:.4f}')
axes[0].set_xscale('log')
axes[0].set_xlabel('Jumlah Trial (log scale)')
axes[0].set_ylabel('P̂(issue > 12 hari)')
axes[0].set_title('Konvergensi Monte Carlo')
axes[0].legend(fontsize=8)

# Kanan — distribusi days_open
vals_sorted, cnts = np.unique(valid_days, return_counts=True)
axes[1].bar(vals_sorted, cnts, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(THRESHOLD, color='tomato', linewidth=2, linestyle='--',
                label=f'Threshold = {THRESHOLD} hari')
axes[1].set_xlabel('days_open (hari)')
axes[1].set_ylabel('Jumlah Issue')
axes[1].set_title(f'Distribusi days_open | vscode (n=590)\n'
                  f'skewness={SKEWNESS}, mean={valid_days.mean():.2f}')
axes[1].legend()

plt.suptitle('Monte Carlo — P(issue > 12 hari) | microsoft/vscode',
             fontsize=13, y=1.02)
plt.tight_layout()
os.makedirs(os.path.join(ROOT, 'report'), exist_ok=True)
plt.savefig(os.path.join(ROOT, 'report', 'fig_mc_convergence.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan → report/fig_mc_convergence.png')

### Interpretasi — Monte Carlo

##### Simulasi Monte Carlo dengan 50.000 trial menghasilkan estimasi P̂(issue > 12 hari) = 0.0000, yang sepenuhnya konsisten dengan baseline empiris yang dihitung Member A (Multazam) dari 590 issues: seluruh 590 issues (100%) berhasil ditutup dalam rentang 0 hingga 12 hari, dengan tidak satu pun yang melewati batas threshold.

##### Hasil ini menunjukkan bahwa repository microsoft/vscode memiliki proses triage dan penanganan issue yang sangat responsif. Dengan median days_open = 0 hari dan mean = 1.68 hari, sebagian besar issue diselesaikan dalam hari yang sama atau keesokan harinya — mencerminkan tim maintainer yang aktif dan terorganisir dengan baik.

---
## 3. Bloom Filter — Deduplication PR Event Stream

### Latar Belakang & Motivasi

Bloom Filter adalah struktur data probabilistik yang **efisien secara memori**
untuk menguji keanggotaan suatu elemen dalam sebuah himpunan. Sifatnya:
- **Tidak pernah false negative** → jika item tidak ada, pasti dilaporkan tidak ada
- **Bisa false positive** → item tidak ada bisa dilaporkan ada, dengan probabilitas = FPR teoritis

**Motivasi dari Member D:**
Z-test dua sampel (Z = 5.4433, p ≈ 0.0000) membuktikan merge rate **User (89.86%)** dan
**Bot (70.45%)** berbeda secara signifikan.
Dalam pipeline CI/monitoring dengan **1.300 PR**, Bloom Filter digunakan untuk
**mendeteksi duplikat event secara efisien** — memastikan setiap PR hanya diproses
sekali tanpa menyimpan seluruh ID dalam memori.

**Formula FPR teoritis (Tsun, 2020, p. 329):**

$$FPR = \left(1 - \left(1 - \frac{1}{m}\right)^n\right)^k$$

di mana $m$ = ukuran bit array, $n$ = jumlah item dimasukkan, $k$ = jumlah hash function.

In [ ]:
# ── Parameter Bloom Filter ───────────────────────────────────────────────
N_ITEMS = len(df_pulls)     # 1.300 PR
K_HASH  = 3                 # jumlah hash function
M_BITS  = N_ITEMS * 10      # 13.000 bit (~10× jumlah item, aturan praktis)

print('=== Parameter Bloom Filter ===')
print(f'  Jumlah item (n)      : {N_ITEMS:,}  (PR di vscode)')
print(f'  Jumlah hash (k)      : {K_HASH}')
print(f'  Ukuran bit array (m) : {M_BITS:,}')

# ── Inisialisasi & insert semua nomor PR ─────────────────────────────────
bf      = BloomFilter(k=K_HASH, m=M_BITS)
pr_ids  = df_pulls['number'].astype(str).tolist()
for pr_id in pr_ids:
    bf.add(pr_id)

fpr_aktual = bf.theoretical_fpr(N_ITEMS)
print()
print(bf)
print(f'  FPR teoritis (n={N_ITEMS:,}) : {fpr_aktual:.6f}  ({fpr_aktual*100:.4f}%)')

In [ ]:
# ── Uji Membership ────────────────────────────────────────────────────────
sample_ada   = pr_ids[:5]                           # pasti ADA di dataset
sample_palsu = [f'FAKE_PR_{i}' for i in range(5)]  # pasti TIDAK ADA

print('=== Uji Membership Bloom Filter ===')
print()
print('PR yang ADA di dataset (harus semua FOUND):')
for sid in sample_ada:
    ok   = bf.contains(sid)
    ikon = '✓ FOUND' if ok else '✗ MISSED — FALSE NEGATIVE (tidak boleh terjadi!)'
    print(f'  PR #{sid:<8} → {ikon}')

print()
print('PR PALSU yang TIDAK ADA (harapan: Correctly rejected):')
fp = 0
for sid in sample_palsu:
    ok   = bf.contains(sid)
    if ok:
        fp += 1
    ikon = '✗ FALSE POSITIVE' if ok else '✓ Correctly rejected'
    print(f'  {sid:<16} → {ikon}')

print()
print(f'False positive dalam sampel 5 item  : {fp}/5')
print(f'FPR teoritis keseluruhan (n=1.300)  : {fpr_aktual:.6f}')

In [ ]:
# ── Visualisasi ──────────────────────────────────────────────────────────
n_vals   = np.arange(1, N_ITEMS + 1, max(1, N_ITEMS // 200))
fpr_vals = [bf.theoretical_fpr(n) for n in n_vals]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Kiri — FPR teoritis vs n
axes[0].plot(n_vals, fpr_vals, color='darkorange', linewidth=2)
axes[0].axvline(N_ITEMS, color='steelblue', linestyle='--',
                label=f'n aktual = {N_ITEMS:,} PR')
axes[0].axhline(fpr_aktual, color='tomato', linestyle=':',
                label=f'FPR = {fpr_aktual:.6f}')
axes[0].set_xlabel('Jumlah Item (n)')
axes[0].set_ylabel('FPR Teoritis')
axes[0].set_title(f'Bloom Filter: FPR vs n\n(k={K_HASH}, m={M_BITS:,})')
axes[0].legend()
axes[0].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=3))

# Kanan — PR per author_type (motivasi deduplication)
ct = df_pulls.groupby('author_type')['is_merged'].value_counts().unstack(fill_value=0)
ct.plot(kind='bar', ax=axes[1], color=['#DD8452', '#4C72B0'],
        edgecolor='white')
axes[1].set_xlabel('Tipe Kontributor')
axes[1].set_ylabel('Jumlah PR')
axes[1].set_title(
    f'PR per Tipe Kontributor\n'
    f'User θ̂={THETA_USER} (n=1124)  |  Bot θ̂={THETA_BOT} (n=176)\n'
    f'Z={Z_STAT} → Tolak H₀'
)
axes[1].legend(title='Merged', labels=['Tidak Merged', 'Merged'])
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Bloom Filter — Deduplication PR Event Stream | microsoft/vscode',
             fontsize=12, y=1.03)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'report', 'fig_bloom_fpr.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan → report/fig_bloom_fpr.png')

### Interpretasi — Bloom Filter

> ⚠️ **[TULIS INTERPRETASIMU SENDIRI DI SINI — TIDAK BOLEH DIBUAT OLEH AI]**
>
> Panduan pertanyaan yang harus kamu jawab:
> - Berapa FPR teoritis pada n = 1.300 PR? Apakah nilai ini dapat diterima untuk use case deduplication monitoring?
> - Mengapa Bloom Filter dipilih dibanding `set()` Python biasa? Apa trade-off konkretnya untuk skala data vscode?
> - Grafik kiri: apa yang terjadi pada FPR saat n terus bertambah? Kapan kita perlu resize filter?
> - Grafik kanan: bagaimana perbedaan merge rate User vs Bot (Z=5.44 dari Member D) relevan dengan kebutuhan deduplication?
> - Apa rekomendasi konkret untuk maintainer vscode terkait Bloom Filter?

---
## 4. MCMC Knapsack — Prioritisasi Intervensi Repository

### Latar Belakang & Motivasi

MCMC digunakan untuk mendekati solusi **0/1 Knapsack**:
pilih subset intervensi yang **memaksimalkan total dampak** tanpa melebihi
**kapasitas effort** satu sprint.

**Motivasi dari seluruh temuan kelompok:**

| Temuan | Sumber | Implikasi Intervensi |
|---|---|---|
| Bot merge rate hanya 70.45% | Member D | Perlu quality gate PR Bot |
| User merge rate 89.86% | Member D | Proses review User sudah baik, pertahankan |
| Semua issue selesai ≤ 12 hari | Member A + RQ3 | Triage sudah sangat responsif |
| Volume PR tinggi: 1.300 | Member A | Risiko reviewer overload |
| CI merge rate 95% sempit | Member C | Estimasi stabil, perlu dijaga |

**Mengapa MCMC, bukan Dynamic Programming?**
DP optimal untuk N kecil. MCMC dipilih karena:
(1) mudah diperluas ke ratusan intervensi tanpa mengubah kode,
(2) dapat dimodifikasi untuk reward probabilistik,
(3) sesuai materi Tsun (2020) p. 317–320.

**Algoritma (Tsun, 2020, p. 317–320):**
1. State awal: semua item tidak dipilih (vektor biner {0,1}^N)
2. Proposal: flip satu bit acak (tambah/hapus satu item)
3. Tolak langsung jika tidak feasible (bobot > kapasitas)
4. Terima jika nilai proposal ≥ nilai saat ini
5. Simpan state terbaik yang pernah ditemui

In [ ]:
# ── Item intervensi berdasarkan temuan seluruh kelompok ──────────────────
# Format: (nama, bobot_effort_hari_orang, nilai_dampak)
#
# Bobot = estimasi effort tim (hari-orang per sprint)
# Nilai  = estimasi dampak terhadap kesehatan repository (skor 1–30)

items = [
    # (nama,                                         bobot, nilai)
    ('Bot PR quality gate (CI lint+test wajib)',         15,    28),  # atasi Bot merge rate 70.45%
    ('Filter/validasi PR Bot sebelum masuk review',       8,    20),  # efisiensi reviewer — Member D
    ('SLA review <= 12 jam untuk PR baru',               12,    25),  # jaga response time issue
    ('Otomasi label issue (auto-triage)',                  6,    15),  # dari EDA Member A
    ('Review latency monitoring dashboard',              10,    18),  # cegah overload 1300+ PR
    ('Template PR standar kontributor baru',              5,    12),  # tingkatkan kualitas PR
    ('Notifikasi stale PR (>30 hari)',                    4,    10),  # kebersihan backlog
    ('Panduan kontributor (CONTRIBUTING.md)',              7,    14),  # onboarding community
]

CAPACITY = 35   # kapasitas sprint dalam hari-orang

print('=== Item Intervensi | microsoft/vscode ===')
print(f'  {"Nama":<48} {"Bobot":>6} {"Nilai":>6}')
print('  ' + '-'*63)
for name, w, v in items:
    print(f'  {name:<48} {w:>6} {v:>6}')
print()
print(f'  Kapasitas sprint : {CAPACITY} hari-orang')
print(f'  Total nilai max  : {sum(v for _,_,v in items)}')
print(f'  Total bobot max  : {sum(w for _,w,_ in items)} (lebih besar dari kapasitas → knapsack bermakna)')

In [ ]:
# ── Jalankan MCMC (src/simulation.py) ────────────────────────────────────
kp = mcmc_knapsack(items=items, capacity=CAPACITY, n_iter=100_000)

print('=== Hasil MCMC Knapsack ===')
print()
print('Item terpilih:')
for item in kp['best_items']:
    print(f'  ✓ {item}')
print()
print(f'  Total nilai    : {kp["best_value"]}')
print(f'  Total bobot    : {kp["best_weight"]} / {CAPACITY} hari-orang')
print(f'  Sisa kapasitas : {CAPACITY - kp["best_weight"]} hari-orang')
print(f'  Accept rate    : {kp["accept_rate"]:.4f}')
print(f'  Iterasi        : {kp["n_iter"]:,}')

# Feasibility check
assert kp['best_weight'] <= CAPACITY, 'ERROR: bobot melebihi kapasitas!'
print()
print('Feasibility check: PASSED ✓')

In [ ]:
# ── Visualisasi hasil MCMC Knapsack ──────────────────────────────────────
selected = set(kp['best_items'])
names    = [it[0] for it in items]
bobs     = [it[1] for it in items]
vals     = [it[2] for it in items]
colors   = ['steelblue' if n in selected else 'lightgray' for n in names]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Kiri — nilai per item
bars = axes[0].barh(names, vals, color=colors)
axes[0].set_xlabel('Nilai Dampak')
axes[0].set_title('Nilai per Intervensi\n(biru = dipilih MCMC | abu = tidak dipilih)')
axes[0].invert_yaxis()

# Kanan — bobot per item
axes[1].barh(names, bobs, color=colors)
axes[1].axvline(CAPACITY, color='tomato', linestyle='--',
                label=f'Kapasitas = {CAPACITY} hari-orang')
axes[1].set_xlabel('Bobot Effort (hari-orang)')
axes[1].set_title('Bobot per Intervensi\n(biru = dipilih MCMC | abu = tidak dipilih)')
axes[1].invert_yaxis()
axes[1].legend()

plt.suptitle(
    f'MCMC Knapsack — Prioritisasi Intervensi | microsoft/vscode\n'
    f'Nilai={kp["best_value"]}  Bobot={kp["best_weight"]}/{CAPACITY}  '
    f'Accept rate={kp["accept_rate"]:.4f}',
    fontsize=12, y=1.04
)
plt.tight_layout()
plt.savefig(os.path.join(ROOT, 'report', 'fig_mcmc_knapsack.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Plot disimpan → report/fig_mcmc_knapsack.png')

### Interpretasi — MCMC Knapsack

> ⚠️ **[TULIS INTERPRETASIMU SENDIRI DI SINI — TIDAK BOLEH DIBUAT OLEH AI]**
>
> Panduan pertanyaan yang harus kamu jawab:
> - Intervensi mana yang dipilih MCMC? Apakah masuk akal dikaitkan dengan temuan Member D (Bot merge rate 70.45%, Z=5.44)?
> - Berapa accept rate? Apa artinya dalam konteks konvergensi algoritma MCMC?
> - Mengapa MCMC dipilih dibanding DP untuk kasus ini? Apa kelebihannya untuk konteks masa depan (ratusan intervensi)?
> - Bagaimana hasil ini berkaitan langsung dengan temuan dari Member A (EDA), B (Estimation), C (CI), D (Z-test)?
> - Tuliskan ≥ 2 rekomendasi **spesifik dan actionable** untuk maintainer vscode berdasarkan hasil MCMC ini.

---
## 5. Ringkasan Temuan & Keterkaitan Antar Layer

In [ ]:
summary = pd.DataFrame({
    'Teknik': ['Monte Carlo', 'Bloom Filter', 'MCMC Knapsack'],
    'Pertanyaan / Tujuan': [
        f'RQ3: P(issue > {THRESHOLD} hari) — tanpa asumsi distribusi',
        'Deduplication PR event stream (1.300 PR, User vs Bot)',
        'Prioritisasi intervensi — keterbatasan 35 hari-orang/sprint'
    ],
    'Hasil Utama': [
        f"P̂={mc_result['p_hat']:.4f}  SE={mc_result['std_error']:.6f}",
        f"FPR={fpr_aktual:.6f}  (k={K_HASH}, m={M_BITS:,})",
        f"Nilai={kp['best_value']}  Bobot={kp['best_weight']}/{CAPACITY}"
    ],
    'Justifikasi Metode': [
        f'Skewness={SKEWNESS} → bukan normal → analitik bias',
        f'Volume tinggi, hemat memori vs set(); motivasi Z={Z_STAT}',
        'Scalable, tidak butuh solusi optimal global'
    ],
    'Referensi Tsun': ['p. 314–315', 'p. 329', 'p. 317–320']
})

print(summary.to_string(index=False))

### Kesimpulan — Computational Analysis

> ⚠️ **[TULIS KESIMPULANMU SENDIRI DI SINI — TIDAK BOLEH DIBUAT OLEH AI]**
>
> Panduan (jawab semua poin ini):
> 1. **Monte Carlo** — ringkas P̂ dan hubungkan ke baseline Member A (skewness 1.69, P_empiris=0.0000)
> 2. **Bloom Filter** — ringkas FPR dan hubungkan ke skala PR (1.300) dan perbedaan Bot/User dari Member D
> 3. **MCMC** — ringkas intervensi terpilih, hubungkan ke θ̂ (B), CI (C), Z-test (D)
> 4. **≥ 2 Rekomendasi** — spesifik, actionable, bisa dibaca langsung oleh maintainer vscode
> 5. **Justifikasi** mengapa ketiga metode ini adalah alat yang *tepat* untuk konteks repository ini

---

**Catatan untuk laporan** `report/statistical_health_report.pdf`:
- Section **"Analisis Komputasional"** (maks 2 halaman) → gunakan temuan dan tabel ringkasan di atas
- Section **"Rekomendasi"** (maks 1 halaman) → gunakan ≥ 2 rekomendasi dari MCMC
- Tiga gambar tersimpan di `report/` siap digunakan sebagai ilustrasi